# BMI Neural Network from Scratch

This notebook demonstrates building and training a simple neural network for BMI classification using a custom implementation in Python. Sections include configuration, data loading, feature engineering, training, and prediction.

## 1. Configuration & Hyperparameters
Set up random seed, learning rate, epochs, and reporting interval.

In [18]:
# Import required libraries
import math
import random
import pandas as pd
from pathlib import Path

# Configuration / hyperparameters
random_seed = 42      # Ensures reproducibility of results
learning_rate = 0.1   
epochs = 10000        
report_every = 1000   

random.seed(random_seed)  # Set random seed for reproducibility

## 2. Data Loading
Load BMI data from CSV and check for file existence.

In [19]:
# Define path to the CSV file containing BMI data
csv_path = Path("../bmi.csv")
if not csv_path.exists():
    # Raise error if file is not found
    raise FileNotFoundError(f"CSV file not found at {csv_path}. Adjust path.")

# Load the raw data into a DataFrame
raw_df = pd.read_csv(csv_path, header=0, names=["Gender", "Height", "Weight", "Index"])
raw_df.head()

,Gender,Height,Weight,Index
0,Male,174,96,4
1,Male,189,87,2
2,Female,185,110,4
3,Female,195,104,3
4,Male,149,61,3


## 3. Feature Engineering
Transform raw features for neural network input and scale target variable.

In [20]:
# Create a new DataFrame for processed features
processed_df = pd.DataFrame()
# Encode gender: 0.1 for male, 0.2 for female (arbitrary encoding for input)
processed_df["Gender"] = raw_df["Gender"].apply(lambda g: 0.1 if str(g).lower() == "male" else 0.2)
# Scale height and weight for better numerical stability
processed_df["Height"] = raw_df["Height"] / 1000.0
processed_df["Weight"] = raw_df["Weight"] / 100.0
# Scale index to [0, 1] range
processed_df["Index"]  = raw_df["Index"] / 10.0

# Calculate mean and standard deviation for normalization
height_mean = processed_df["Height"].mean()
height_std  = processed_df["Height"].std()
weight_mean = processed_df["Weight"].mean()
weight_std  = processed_df["Weight"].std()

# Check for zero standard deviation to avoid division by zero
if height_std == 0 or weight_std == 0:
    raise ValueError("Standard deviation of height or weight is zero; cannot standardize.")

# Find min and max of index for scaling target
index_min = processed_df["Index"].min()
index_max = processed_df["Index"].max()
index_range = index_max - index_min
if index_range == 0:
    raise ValueError("Index range is zero; cannot scale target.")

processed_df.head()  # Show processed features

,Gender,Height,Weight,Index
0,0.1,0.174,0.96,0.4
1,0.1,0.189,0.87,0.2
2,0.2,0.185,1.10,0.4
3,0.2,0.195,1.04,0.3
4,0.1,0.149,0.61,0.3


## 4. Build Training Data
Prepare normalized input vectors and scaled targets for training.

In [21]:
# Build training data: normalize inputs and scale target
training_data = []
for _, row in processed_df.iterrows():
    gender_input = row["Gender"]  # Encoded gender
    # Normalize height and weight using mean and std
    height_norm = (row["Height"] - height_mean) / height_std
    weight_norm = (row["Weight"] - weight_mean) / weight_std
    # Scale target index to [0.1, 0.9] for sigmoid output
    scaled_target_value = 0.1 + 0.8 * (row["Index"] - index_min) / index_range
    training_data.append({
        "inputs": [gender_input, height_norm, weight_norm],
        "target": scaled_target_value
    })
# Show first few samples for inspection
training_data[:3]

[{'inputs': [np.float64(0.1),
   np.float64(0.2476907134097988),
   np.float64(-0.3088077453112467)],
  'target': np.float64(0.7400000000000001)},
 {'inputs': [np.float64(0.1),
   np.float64(1.1637066653691137),
   np.float64(-0.5867347160913684)],
  'target': np.float64(0.42000000000000004)},
 {'inputs': [np.float64(0.2),
   np.float64(0.9194357448466298),
   np.float64(0.12352309812449869)],
  'target': np.float64(0.7400000000000001)}]

## 5. Network Setup
Initialize weights and biases for a single hidden layer neural network.

In [ ]:
# Define network architecture
n_inputs = 3      # Number of input features (gender, height, weight)
n_hidden = 5     # Number of hidden neurons (arbitrary, can be tuned)
n_outputs = 1    # Single output for regression

# Initialize weights and biases with random values
weight_input_to_hidden = [
    [random.uniform(-0.5, 0.5) for _ in range(n_inputs)]
    for _ in range(n_hidden)
]
bias_hidden = [random.uniform(-0.5, 0.5) for _ in range(n_hidden)]
weight_hidden_to_output = [random.uniform(-0.5, 0.5) for _ in range(n_hidden)]
bias_output = random.uniform(-0.5, 0.5)

## 6. Helper Functions
Sigmoid activation and its derivative for stability.

In [23]:
# Sigmoid activation function (keeps output in [0,1])
def _sigmoid(x):
    # Clamp x for numerical stability
    x = max(min(x, 500), -500)
    return 1.0 / (1.0 + math.exp(-x))

# Derivative of sigmoid (used for backpropagation)
def _sigmoid_derivative(y):
    return y * (1.0 - y)

## 7. Training Loop
Train the neural network and report progress at intervals.

In [24]:
print("--- Starting Network Training ---")
for epoch in range(epochs):
    sum_squared_error = 0.0  # Accumulate squared error for reporting

    for sample in training_data:
        input_vector = sample["inputs"]
        target_value = sample["target"]

        # Forward pass: compute hidden layer activations
        hidden_activations = [0.0] * n_hidden
        for h in range(n_hidden):
            acc = 0.0
            for i in range(n_inputs):
                acc += input_vector[i] * weight_input_to_hidden[h][i]  # Weighted sum
            acc -= bias_hidden[h]  # Subtract bias
            hidden_activations[h] = _sigmoid(acc)  # Apply activation

        # Forward pass: compute output layer activation
        output_net = 0.0
        for h in range(n_hidden):
            output_net += hidden_activations[h] * weight_hidden_to_output[h]
        output_net -= bias_output  # Subtract output bias
        output_activation = _sigmoid(output_net)

        # Compute error (difference between target and prediction)
        error = target_value - output_activation
        sum_squared_error += error * error  # Accumulate squared error

        # Backpropagation: calculate output delta
        output_delta = error * _sigmoid_derivative(output_activation)

        # Backpropagation: calculate hidden deltas
        hidden_deltas = [0.0] * n_hidden
        for h in range(n_hidden):
            hidden_deltas[h] = output_delta * weight_hidden_to_output[h] * _sigmoid_derivative(hidden_activations[h])

        # Update weights and biases: hidden to output
        for h in range(n_hidden):
            weight_hidden_to_output[h] += learning_rate * output_delta * hidden_activations[h]
        bias_output -= learning_rate * output_delta

        # Update weights and biases: input to hidden
        for h in range(n_hidden):
            for i in range(n_inputs):
                weight_input_to_hidden[h][i] += learning_rate * hidden_deltas[h] * input_vector[i]
        for h in range(n_hidden):
            bias_hidden[h] -= learning_rate * hidden_deltas[h]

    # Report progress at intervals
    if (epoch % report_every == 0) or (epoch == epochs - 1):
        print(f"> Epoch={epoch}, Learning Rate={learning_rate:.2f}, Error={sum_squared_error:.4f}")

print("--- Training Complete ---")

--- Starting Network Training ---
> Epoch=0, Learning Rate=0.10, Error=21.1354
> Epoch=1000, Learning Rate=0.10, Error=1.3608
> Epoch=1000, Learning Rate=0.10, Error=1.3608
> Epoch=2000, Learning Rate=0.10, Error=1.3033
> Epoch=2000, Learning Rate=0.10, Error=1.3033
> Epoch=3000, Learning Rate=0.10, Error=1.2547
> Epoch=3000, Learning Rate=0.10, Error=1.2547
> Epoch=4000, Learning Rate=0.10, Error=1.1660
> Epoch=4000, Learning Rate=0.10, Error=1.1660
> Epoch=5000, Learning Rate=0.10, Error=1.0525
> Epoch=5000, Learning Rate=0.10, Error=1.0525
> Epoch=6000, Learning Rate=0.10, Error=0.9989
> Epoch=6000, Learning Rate=0.10, Error=0.9989
> Epoch=7000, Learning Rate=0.10, Error=0.9850
> Epoch=7000, Learning Rate=0.10, Error=0.9850
> Epoch=8000, Learning Rate=0.10, Error=0.9795
> Epoch=8000, Learning Rate=0.10, Error=0.9795
> Epoch=9000, Learning Rate=0.10, Error=0.9760
> Epoch=9000, Learning Rate=0.10, Error=0.9760
> Epoch=9999, Learning Rate=0.10, Error=0.9731
--- Training Complete ---
> 

## 8. Prediction Example
Use the trained network to predict BMI category for a sample input.

In [25]:
# Example prediction: input values for a new person
gender_raw_input = 1    # 1 for male
height_cm_input = 189   # Height in centimeters
weight_kg_input = 87    # Weight in kilograms

# Encode and scale input features using training statistics
gender_encoded = 0.1 if gender_raw_input == 1 else 0.2
height_scaled_input = ((height_cm_input / 1000.0) - height_mean) / height_std
weight_scaled_input = ((weight_kg_input / 100.0) - weight_mean) / weight_std
prediction_input_vector = [gender_encoded, height_scaled_input, weight_scaled_input]

# Forward pass through network for prediction
hidden_activations_pred = [0.0] * n_hidden
for h in range(n_hidden):
    acc = 0.0
    for i in range(n_inputs):
        acc += prediction_input_vector[i] * weight_input_to_hidden[h][i]
    acc -= bias_hidden[h]
    hidden_activations_pred[h] = _sigmoid(acc)

output_net_pred = 0.0
for h in range(n_hidden):
    output_net_pred += hidden_activations_pred[h] * weight_hidden_to_output[h]
output_net_pred -= bias_output
scaled_output_pred = _sigmoid(output_net_pred)

# Unscale output to original index space
unscaled_index_over_10 = (scaled_output_pred - 0.1) / 0.8
index_original_space = unscaled_index_over_10 * index_range + index_min
predicted_class_index = round(index_original_space * 10)

# Map predicted index to BMI category
if predicted_class_index <= 0:
    predicted_category = "Extremely Weak"
elif predicted_class_index == 1:
    predicted_category = "Weak"
elif predicted_class_index == 2:
    predicted_category = "Normal"
elif predicted_class_index == 3:
    predicted_category = "Overweight"
elif predicted_class_index == 4:
    predicted_category = "Obesity"
else:
    predicted_category = "Extreme Obesity"

# Print prediction results
print("\n--- Prediction ---")
print(f"Scaled network output: {scaled_output_pred:.4f}")
print(f"Predicted Index (value/10): {index_original_space:.3f}")
print(f"Predicted Class: {predicted_class_index}")
print(f"Category: {predicted_category}")


--- Prediction ---
Scaled network output: 0.4790
Predicted Index (value/10): 0.237
Predicted Class: 2
Category: Normal
